In [18]:
import numpy as np
import cv2
from collections import Counter

In [28]:
#1. Выполните сохранение монохромного изображения в виде текстового или бинарного файла.

image_path = 'sar_1_gray.jpg'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
image = image.astype(np.float32)
image.tofile('bin_test.bin')
original_size = image.nbytes
print(f"Размер исходного изображения: {original_size} байт")

Размер исходного изображения: 960000 байт


In [29]:
#2. Реализуйте алгоритм вейвлет-преобразования Хаара для изображения.

def haar_transform(img):
    h, w = img.shape
    temp = np.zeros_like(img, dtype=np.float32)
    for i in range(h):
        for j in range(0, w, 2):
            if j + 1 < w:
                temp[i, j // 2] = (img[i, j] + img[i, j + 1]) / 2
                temp[i, j // 2 + w // 2] = (img[i, j] - img[i, j + 1]) / 2
    result = np.zeros_like(temp, dtype=np.float32)
    for j in range(w):
        for i in range(0, h, 2):
            if i + 1 < h:
                result[i // 2, j] = (temp[i, j] + temp[i + 1, j]) / 2
                result[i // 2 + h // 2, j] = (temp[i, j] - temp[i + 1, j]) / 2
    ll = result[:h // 2, :w // 2]
    hl = result[h // 2:, :w // 2]
    lh = result[:h // 2, w // 2:]
    hh = result[h // 2:, w // 2:]
    return ll, lh, hl, hh

ll, lh, hl, hh = haar_transform(image)

In [31]:
#3. Выполните квантование высокочастотных компонент (прим., количество квантов = 4).

n_quants = 4
def quantize(coeffs, n_quants):
    min_val = np.min(coeffs)
    max_val = np.max(coeffs)
    step = (max_val - min_val) / n_quants if max_val != min_val else 1
    quantized = np.round((coeffs - min_val) / step).astype(int)
    return quantized, min_val, step

lh_q, lh_min, lh_step = quantize(lh, n_quants)
hl_q, hl_min, hl_step = quantize(hl, n_quants)
hh_q, hh_min, hh_step = quantize(hh, n_quants)

In [32]:
#4. Сохраните получившийся массив значений в текстовый или бинарный файл в порядке LL, LH, HL, HH вейвлет-преобразования Хафа. Компоненты LH, HL, HH храните в виде пар (значение, количество повторений).

def run_length_encode(data):
    encoded = []
    flat = data.flatten()
    if len(flat) == 0:
        return encoded
    prev_val = flat[0]
    count = 1
    for val in flat[1:]:
        if val == prev_val:
            count += 1
        else:
            encoded.append((prev_val, count))
            prev_val = val
            count = 1
    encoded.append((prev_val, count))
    return encoded

lh_rle = run_length_encode(lh_q)
hl_rle = run_length_encode(hl_q)
hh_rle = run_length_encode(hh_q)

with open('txt_test.txt', 'w') as f:
    f.write("LL\n")
    np.savetxt(f, ll, fmt='%d')
    f.write("\nLH (value, count)\n")
    for value, count in lh_rle:
        f.write(f"{value} {count}\n")
    f.write("\nHL (value, count)\n")
    for value, count in hl_rle:
        f.write(f"{value} {count}\n")
    f.write("\nHH (value, count)\n")
    for value, count in hh_rle:
        f.write(f"{value} {count}\n")

In [34]:
#5. Сравните объем памяти, занимаемый исходным изображением (попиксельное хранение), и изображение, полученным после преобразования Хафа и сжатием длин серий.

print(f"Размер исходного изображения: {original_size} байт")
print(f"Размер после Хаара и RLE: {compressed_size} байт")
print(f"Коэффициент сжатия: {original_size / compressed_size:.2f}x")


Размер исходного изображения: 960000 байт
Размер после Хаара и RLE: 290155 байт
Коэффициент сжатия: 3.31x
